# Finding anomalies in BlueGene/L logs: LogBERT against a rule-guided language model

**What this notebook is.** The results of the full run of the NLP project, with the tables, the charts and the reasoning behind them in one place.
Two approaches were compared on the same data, split and evaluation:

* **Approach A: LogBERT.** A small transformer that is trained on normal log windows only and flags a window when it cannot predict the log keys that were hidden in it.
* **Approach B: a language model guided by rules.** Qwen2.5-7B-Instruct is *not* trained. It is prompted to classify log templates, in four conditions, the last of which does not use the model at all.

**Everything here is read from the files of the run** (`results/` and `docs/figures/`, written by `make results` and `make figures`), so no number is typed in by hand.
The notebook has been run already, so the outputs are saved in it. To run it again, open it in Jupyter from the `NLP/notebooks/` folder: it needs `results/` and `docs/figures/` next to it.

**Contents:**
1. The data and the split · 2. The two approaches · 3. Results at a glance · 4. Approach A in detail · 5. Approach B in detail · 6. Why the methods differ · 7. What each method costs · 8. Conclusions and limits · 9. How to reproduce

In [ ]:
# Where the run is: the folder that holds results/ (this notebook lives in NLP/notebooks/).
here = Path.cwd().resolve()
ROOT = next((p for p in (here, here.parent, here.parent.parent) if (p / "results" / "comparison.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("results/comparison.csv not found: open this notebook from the NLP/ or NLP/notebooks/ folder")
RUN, BASE, FIG = ROOT / "results", None, ROOT / "docs" / "figures"
RUN_LABEL, BASE_LABEL = "this run", None
print("run folder:", RUN)

In [ ]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True, "font.size": 9,
})

CODE = {"logbert": "A", "logbert_full": "A+", "llm_zero_shot": "B1", "llm_few_shot": "B2",
        "llm_rule_based": "B3", "keyword_rules": "B4"}
WHAT = {"A": "LogBERT", "A+": "LogBERT, full-coverage masking", "B1": "LLM zero-shot",
        "B2": "LLM few-shot", "B3": "LLM + rules", "B4": "keyword rules (no LLM)"}
BLUE, ORANGE, GREY = "#2f6db5", "#e07b24", "#8a8a85"


def seconds_text(s):
    if s is None or pd.isna(s):
        return "-"
    if s < 1:
        return f"{s * 1000:.0f} ms"
    if s < 120:
        return f"{s:.0f} s"
    if s < 7200:
        return f"{s / 60:.0f} min"
    return f"{s / 3600:.1f} h"


def read_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else default


def comparison_of(run):
    """The comparison table of a run folder (results/ layout), with the method codes A ... B4."""
    table = pd.read_csv(Path(run) / "comparison.csv")
    table["code"] = table["method"].map(CODE)
    table["what"] = table["code"].map(WHAT)
    return table


def plot_history(run, title="LogBERT training"):
    """Loss per epoch of a run folder that holds logbert/history.json."""
    history = pd.DataFrame(read_json(Path(run) / "logbert" / "history.json"))
    best = history.loc[history["dev_loss"].idxmin()]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    for ax, (name, key) in zip(axes, [("total loss", "loss"), ("masked key prediction (MLKP)", "mlkp"),
                                      ("hypersphere distance (VHM)", "vhm")]):
        ax.plot(history["epoch"], history[f"train_{key}"], marker="o", ms=3, color=BLUE, label="train")
        ax.plot(history["epoch"], history[f"dev_{key}"], marker="o", ms=3, color=ORANGE,
                label="dev (held-out normal windows)")
        ax.axvline(best["epoch"], color=GREY, ls=":")
        ax.set_title(name)
        ax.set_xlabel("epoch")
    axes[0].legend(frameon=False)
    fig.suptitle(title, y=1.02)
    plt.show()
    print(f"{len(history)} epochs; best dev loss {best['dev_loss']:.4f} at epoch {int(best['epoch'])} "
          f"(dotted line); {history['seconds'].mean():.0f} s per epoch, {history['seconds'].sum() / 60:.1f} min in total")
    return history


def logbert_summary(run):
    """Tuned settings and validation / test scores of A and A+ (reads results/<method>.json of a run folder)."""
    rows = []
    for method in ("logbert", "logbert_full"):
        found = read_json(Path(run) / f"{method}.json") or read_json(Path(run) / "results" / f"{method}.json")
        if not found:
            continue
        rows.append({"method": CODE[method], "g": found["settings"]["g"], "r": found["settings"]["r"],
                     "validation F1": found["validation"]["f1"], "test precision": found["test"]["precision"],
                     "test recall": found["test"]["recall"], "test F1": found["test"]["f1"],
                     "test false positives": found["test"]["fp"]})
    return pd.DataFrame(rows)

## 1. The data and the split

**BGL** (BlueGene/L) is a supercomputer log from Lawrence Livermore National Laboratory, in the Loghub collection: 4,747,963 messages over 32 weeks
(2005-05-30 to 2006-01-02). Every message is either normal or carries an alert tag (a category such as `KERNMC` or `APPSEV`) that marks a fault.

How the data is prepared, the same for both approaches:

| Step | What is done |
|---|---|
| Templates | Drain turns each message into a template by replacing the variable parts (numbers, addresses) with `<*>`. The vocabulary of "log keys" is the set of templates. |
| Split | **By time, never at random**: the earliest 60% of the timeline is for training, the next 20% for validation (tuning), the last 20% for testing. |
| Windows | 5-minute windows that slide by 1 minute. A window is *anomalous* if it holds at least one alert message. |
| Sizes | validation: 10,478 windows, 446 anomalous (4.3%) · test: 5,742 windows, 495 anomalous (8.6%) |

**The fact that shapes every result below:** the log changes over time. Messages whose template **never occurred in the training period** are
**8.9%** of the validation period but **79.6%** of the test period (figure 2). A method that learned what "normal" looks like in the training period
meets, in the test period, mostly messages it has never seen.

In [ ]:
GALLERY = {
    1: ("01_bgl_timeline.png", "Figure 1. BGL over time: normal and alert messages per week"),
    2: ("02_unseen_templates.png", "Figure 2. Share of messages whose template never occurred in the training period"),
    3: ("03_logbert_training.png", "Figure 3. LogBERT: loss per epoch"),
    4: ("04_logbert_tuning_grid.png", "Figure 4. LogBERT: validation F1 for every (g, r) setting"),
    5: ("05_logbert_key_misses.png", "Figure 5. LogBERT: which log keys it cannot predict (known templates against never-seen ones)"),
    6: ("06_logbert_window_scores.png", "Figure 6. LogBERT: window scores of normal, unseen-template and anomalous windows"),
    7: ("07_results_pr_space.png", "Figure 7. Every method in precision-recall space"),
    8: ("08_llm_flagged_templates.png", "Figure 8. Templates flagged by each approach B condition"),
    9: ("09_keyword_rule_sensitivity.png", "Figure 9. Keyword rules: how many rules are needed"),
    10: ("10_inference_cost.png", "Figure 10. What each method costs at inference time"),
}


def show_figures(*numbers):
    """Show figures of the run (all ten when no number is given)."""
    shown = 0
    for number in (numbers or sorted(GALLERY)):
        name, caption = GALLERY[number]
        if (FIG / name).exists():
            display(Markdown(f"**{caption}**"))
            display(Image(filename=str(FIG / name)))
            shown += 1
    print(f"{shown} of {len(numbers) or len(GALLERY)} figures shown (from {FIG})")

show_figures(1, 2)

## 2. The two approaches

| | **A. LogBERT** | **B. Rule-guided language model** |
|---|---|---|
| The model | a 2-layer transformer encoder (hidden size 256, 4 attention heads), **trained from scratch** | Qwen2.5-7B-Instruct, **not trained**: it is only given prompts |
| What it learns from | normal training windows only, no alert labels | the *labeled* training templates (to write rules) |
| How it decides | hides half of the log keys of a window and tries to predict them; a window is flagged when more than **r** hidden keys are outside the model's top **g** guesses | classifies each unique **template** as normal or abnormal; a window is flagged when one of its messages is |
| Tuned on validation | g and r (231 settings tried) | nothing: the 30 rules are fixed after induction |
| Variants | **A** as in the paper · **A+** our extension: every key is hidden once, so no key escapes the check | **B1** zero-shot · **B2** few-shot (8 labeled examples) · **B3** with the 30 rules in the prompt · **B4** the 30 rules as plain keyword matching, **no model** |
| This run | 6 epochs of training on a 4-core CPU (60 min), reduced recipe (see section 8) | 4-bit Qwen2.5-7B-Instruct (official `q4_k_m` GGUF) served by llama-cpp-python on the same 4 cores |

**Where the rules come from.** The language model reads batches of labeled training templates and proposes rules (a sentence and a list of keywords). A rule is kept only if it is never wrong on the training templates it matches:
42 prompts produced 160 candidate rules and 99 survived. The top 30 by usage are the rules that B3 and B4 use.

## 3. Results at a glance

Every method is scored on the same test windows with the same code (`scripts/compare.py`). The scores are precision, recall and F1 for the *anomalous* class.

In [ ]:
table = comparison_of(RUN)
n_windows, n_alerts = int(table["support"].iloc[0]), int(table.iloc[0]["tp"] + table.iloc[0]["fn"])
print(f"Test period: {n_windows:,} windows, {n_alerts} of them anomalous ({100 * n_alerts / n_windows:.1f}%)")
best = table.loc[table["f1"].idxmax()]
print(f"Highest F1 on the test windows: {best['code']} {best['what']} ({best['f1']:.3f})")
shown = table[["code", "what", "precision", "recall", "f1", "tp", "fp", "fn", "inference_seconds"]].copy()
shown["inference"] = shown["inference_seconds"].map(seconds_text)
shown = shown.drop(columns="inference_seconds").rename(columns={"code": "", "what": "method"})
display(shown.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}", "tp": "{:,}",
                            "fp": "{:,}", "fn": "{:,}"}).hide(axis="index")
        .background_gradient(subset=["f1"], cmap="Blues", vmin=0, vmax=1))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.8))
x = np.arange(len(table))
for i, (column, label, color) in enumerate([("precision", "precision", BLUE), ("recall", "recall", ORANGE), ("f1", "F1", "#3b3b38")]):
    bars = ax.bar(x + (i - 1) * 0.27, table[column], 0.27, label=label, color=color)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in table[column]], fontsize=7, padding=1)
ax.set_xticks(x, [f"{c}\n{WHAT[c]}" for c in table["code"]], fontsize=8)
ax.set_ylim(0, 1.12)
ax.set_title("Test windows: every method on the same 5-minute windows")
ax.legend(frameon=False, ncol=3, loc="upper left")
plt.show()

**Reading the table.** All methods find most of the anomalous windows: recall runs from 0.90 to 1.00.
They differ in **precision**, that is in how many normal windows they flag by mistake: from 0 false positives (B4) to 4,103 (B1).
That is why the F1 scores are so different, from 0.194 to **0.956** (B4).

* The two LogBERT variants have the best recall among the model-free scores but flag 2,159 / 1,975 of the 5,247 normal windows (A / A+).
* Giving the language model more help raises F1 step by step: zero-shot 0.194 → few-shot 0.481 → with the rules in the prompt 0.644.
* The same 30 rules used as **plain keyword matching, with no model**, give F1 0.956 with 0 false positives, in 8 ms.

In [ ]:
show_figures(7)

## 4. Approach A in detail: LogBERT

LogBERT is trained to predict hidden log keys of **normal** windows (masked key prediction, MLKP) and to pull the representations of normal windows toward a common centre (hypersphere, VHM). Detection hides half of the keys of a window;
the window is flagged if more than **r** of the hidden keys are not among the model's top **g** predictions. g and r are chosen on the validation period.

In [ ]:
display(Markdown("**Approach A: settings chosen on the validation period, and how they do on the test period**"))
summary = logbert_summary(RUN)
display(summary.style.format({"validation F1": "{:.3f}", "test precision": "{:.3f}", "test recall": "{:.3f}",
                              "test F1": "{:.3f}", "test false positives": "{:,}"}).hide(axis="index"))
display(Markdown("**Figure 3. LogBERT: loss per epoch (training and held-out normal windows)**"))
history = plot_history(RUN, "LogBERT training of this run")
if BASE is not None and (BASE / "logbert" / "history.json").exists():
    old = pd.DataFrame(read_json(BASE / "logbert" / "history.json"))
    print(f"{BASE_LABEL}: {len(old)} epochs, {old['seconds'].sum() / 60:.0f} min, best dev loss {old['dev_loss'].min():.4f}")
    print(f"{RUN_LABEL}: {len(history)} epochs, {history['seconds'].sum() / 60:.0f} min, best dev loss {history['dev_loss'].min():.4f}")

**Figure 4. LogBERT: validation F1 for every (g, r) setting**

The heatmaps show the validation F1 for every combination of g (how many top guesses count as a hit) and r (how many misses a window may have). The white circle is the setting that was chosen.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, method in zip(axes, ["logbert", "logbert_full"]):
    path = RUN / "logbert" / f"{method}_tuning_grid.csv"
    if not path.exists():
        ax.set_visible(False)
        continue
    grid = pd.read_csv(path)
    pivot = grid.pivot(index="g", columns="r", values="f1")
    image = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap="viridis", vmin=0)
    ax.set_xticks(range(len(pivot.columns)), pivot.columns, fontsize=7)
    ax.set_yticks(range(len(pivot.index)), pivot.index, fontsize=7)
    best = grid.loc[grid["f1"].idxmax()]
    ax.plot(list(pivot.columns).index(best["r"]), list(pivot.index).index(best["g"]), "o", mfc="none", mec="white", ms=12, mew=2)
    ax.set_xlabel("r: misses allowed")
    ax.set_ylabel("g: top-g candidates")
    ax.set_title(f"{CODE[method]} {WHAT[CODE[method]]}: validation F1 (best {best['f1']:.3f})")
    ax.grid(False)
    fig.colorbar(image, ax=ax, fraction=0.046)
plt.show()

The chosen settings (A: g = 12, r = 3; A+: g = 8, r = 8) reach validation F1 0.599 / 0.614, but the test F1 is 0.293 / 0.305.
That gap is not noise in the tuning: 16 of the 231 settings are within 0.02 of the best validation F1. It is the change in the log between the validation and the test period.

In [ ]:
figure_data = read_json(FIG / "figure_data.json", {})
unseen = figure_data.get("unseen_templates", {}).get("percent_by_split")
panels = figure_data.get("window_scores", {}).get("panels")
misses = figure_data.get("key_misses", {}).get("groups")


def percent(value):
    return "-" if value is None else f"{value:.2f}%"


if unseen:
    print(f"Messages whose template never occurred in training: validation period {unseen['val']:.1f}%, test period {unseen['test']:.1f}%")
if panels:
    print("\nLogBERT (A) on the test windows, by kind of window:")
    for name, values in panels.items():
        print(f"  {name:<58} {values.get('windows', 0):>6,} windows, {percent(values.get('percent_flagged')):>8} flagged")
if misses:
    print("\nLogBERT (A): share of masked keys that are not in the top-g predictions:")
    for name, values in misses.items():
        print(f"  {name:<12} {values.get('keys', 0):>9,} keys, {percent(values.get('percent_missed')):>8} missed")

**Why LogBERT flags so many normal windows in the test period.** Look at the hidden keys it must predict (figure 5). For templates that exist in the training period, it misses
only 1.94% of them (92,859 keys). For templates it has never seen it misses **100%** (360,557 normal keys) and
100% of the 28,094 alert keys: a key that is not in the vocabulary cannot be predicted, so it counts as a miss whether it is harmless or not.

At window level (figure 6): of the normal windows whose templates were all seen in training, only **0.26%** are flagged (1,514 windows);
of the normal windows that contain a template never seen in training, **57.7%** are flagged (3,733 windows); and
**92.1%** of the 495 anomalous windows are flagged. So LogBERT works where the log looks like the training log, and treats "new" as "abnormal".

In [ ]:
show_figures(5, 6)

## 5. Approach B in detail: a language model guided by rules

Each of the 472 unique templates of the test period is classified once and the label is copied to every message that has that template. Then a window is flagged if it has at least one flagged message.

In [ ]:
rules = read_json(RUN / "rules" / "rules.json", [])
info = read_json(RUN / "rules" / "induction_summary.json", {})
if rules:
    print(f"The language model proposed {info.get('n_candidates', '?')} rules in {info.get('n_batches', '?')} prompts; "
          f"{info.get('n_kept', len(rules))} were kept because they are never wrong on the labeled training templates. "
          f"The top 30 are used. Generator: {info.get('generator', '?')}")
    names = {0: "normal", 1: "abnormal", "normal": "normal", "abnormal": "abnormal"}
    overview = pd.DataFrame([{
        "rank": i + 1, "label": names.get(r["label"], r["label"]), "rule (the model's words)": r["text"],
        "keywords": ", ".join(r["keywords"][:4]) + (" ..." if len(r["keywords"]) > 4 else ""),
        "training templates (all correct)": r["correct"], "training messages": r["message_coverage"],
    } for i, r in enumerate(rules[:12])])
    display(Markdown("**The 12 highest-ranked rules**"))
    display(overview.style.format({"training messages": "{:,}"}).hide(axis="index"))
else:
    print("no rules in this run")

In [ ]:
rows = []
for method in ["llm_zero_shot", "llm_few_shot", "llm_rule_based", "keyword_rules"]:
    found = read_json(RUN / f"{method}.json")
    if not found:
        continue
    rows.append({"": CODE[method], "condition": WHAT[CODE[method]], "classifier": found.get("classifier"),
                 "templates classified": found.get("n_templates_classified"),
                 "unparsed answers": found.get("n_unparsed_answers"),
                 "message-level F1": (found.get("test_message_level") or {}).get("f1"),
                 "time to classify": seconds_text(found.get("inference_seconds"))})
if rows:
    display(pd.DataFrame(rows).style.format({"message-level F1": "{:.3f}"}, na_rep="-").hide(axis="index"))

**What the four conditions show.** Without help (B1) the model calls almost everything abnormal: recall 1.00 but 4,103 false positives. Eight examples (B2) cut the false positives to 976.
The rules in the prompt (B3) cut them to 488. The rules as plain keyword matching (B4, no model) have **none**.

**Does B3 really use the rules?** 420 of the 472 answers of B3 cite a rule, but the keyword of the cited rule occurs in the template in only **20** of them.
Of the 37 templates it calls abnormal, only 7 are backed by a rule that occurs in the template. The model quotes rules but mostly does not apply them as written,
which is why the same rules do better when a program applies them (B4).

In [ ]:
show_figures(8, 9)

## 6. Why the methods differ: error analysis of the test period

* **Is labeling templates enough?** 3 of the 472 test templates are *mixed* (they hold both normal and alert messages). They contain only 31 of the 28,915 alert messages.
  A perfect template-level labeler would reach a window F1 of **0.996**. So classifying templates is the right unit for this data, and B4's 0.956 is close to that ceiling.
* **Label drift.** Two templates change their majority label between the training and the test period (table below); a method that learns template labels from the training period cannot get them right.
* **Which alerts are missed.** The heatmap shows the recall per alert category. LogBERT and the keyword rules miss different categories, so the methods are not interchangeable.

In [ ]:
analysis = RUN / "analysis"
rename = {**{k: v for k, v in CODE.items()}}

purity = read_json(analysis / "template_purity.json")
if purity:
    print(f"Templates: {purity['n_templates']} in the test period; {purity['n_mixed_templates']} are mixed (normal and alert messages). "
          f"A perfect template-level labeler would reach window F1 {purity['oracle_window_level']['f1']:.3f}, so labelling templates is enough for this data.")

for title, name in [("Templates whose label changes between training and test", "label_drift.csv"),
                    ("Templates that hold both normal and alert messages", "mixed_templates.csv")]:
    if (analysis / name).exists():
        display(Markdown(f"**{title}**"))
        display(pd.read_csv(analysis / name).style.hide(axis="index"))

if (analysis / "per_category_recall.csv").exists():
    recall = pd.read_csv(analysis / "per_category_recall.csv").rename(columns=rename)
    display(Markdown("**Recall per alert category (window level): which kinds of alerts each method misses**"))
    method_columns = [c for c in recall.columns if c in WHAT]
    display(recall.style.format({c: "{:.2f}" for c in method_columns}).hide(axis="index")
            .background_gradient(subset=method_columns, cmap="Blues", vmin=0, vmax=1))

if (analysis / "method_agreement.csv").exists():
    agree = pd.read_csv(analysis / "method_agreement.csv")
    agree["method_a"], agree["method_b"] = agree["method_a"].map(CODE), agree["method_b"].map(CODE)
    display(Markdown("**Do two methods catch the same anomalous windows? (and do they make the same false alarms?)**"))
    display(agree.style.hide(axis="index"))

if (analysis / "combinations.csv").exists():
    combos = pd.read_csv(analysis / "combinations.csv")
    combos["method_a"], combos["method_b"] = combos["method_a"].map(CODE), combos["method_b"].map(CODE)
    display(Markdown("**Combining an approach A method with an approach B method (either one flags / both flag)**"))
    display(combos.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}", "fp": "{:,}"}).hide(axis="index"))

faith = read_json(analysis / "rule_faithfulness.json")
if faith:
    print("Does the rule-based LLM (B3) apply the rules it cites?")
    print(f"  {faith['answers_citing_a_rule']} of {faith['templates']} answers cite a rule, but the cited rule's keyword occurs in the template "
          f"in only {faith['cited_rule_occurs_in_template']} of them;")
    print(f"  {faith['abnormal_answers']} templates are called abnormal, only {faith['abnormal_answers_backed_by_an_occurring_rule']} of them backed by a rule that really occurs in the template.")

## 7. What each method costs

Inference on the 5,742 test windows (A, A+) or on the 472 test templates (B1 to B3), on 4 CPU cores:

| Method | Inference | One-off cost before it |
|---|---|---|
| A / A+ | 1 min / 3 min | training: 60 min (6 epochs) |
| B1 / B2 / B3 | 47 min / 53 min / 57 min | B3 needs the rules: induction took about 50 min (42 model calls, `results/README.md`) |
| B4 | 8 ms | the same rules |

The language model is the slow part: 57 min to label 472 templates on a CPU, against 1 min for LogBERT to score 5,742 windows. B4 reuses the model's *rules* and needs no model at run time.

In [ ]:
show_figures(10)

## 8. Conclusions and limits

**What the results say**

1. **On this data and split, the best detector is a keyword matcher** whose rules were written by a language model and checked against the training labels: F1 0.956, no false positives, recall 0.915.
   The language model is most useful *before* run time, to propose rules, and less useful as the classifier itself (B1 to B3 over-flag).
2. **LogBERT's low precision is a drift problem, not a tuning problem.** It is tuned to F1 0.599 on validation but reaches 0.293 on test, because 80% of the test messages have templates it has never seen
   (100% of their keys are misses), and 58% of the normal windows with such a template are flagged. Where templates are known it behaves: 0.26% of fully known normal windows are flagged
   and 92% of the anomalous windows are caught.
3. **The model is not the bottleneck for B.** Templates are almost pure (3 mixed of 472); the ceiling for template-level labeling is 0.996. What separates B1 (0.194) from B4 (0.956) is how well the model uses the
   evidence it is given.

**Limits: read the numbers with these in mind**

* **LogBERT here is a reduced-training run.** It ran on a CPU for 6 epochs on de-duplicated chunks (60 min); the training loss was still falling when it stopped
  (2.03 → 0.65) and the best dev loss was at epoch 5. The recipe in `configs/bgl.yaml` allows 50 epochs with early stopping. The conclusion about unseen templates does not depend on the amount of training
  (a key that is not in the vocabulary cannot be predicted), but the exact A and A+ scores would change with a longer run.
* **B4 uses labels, A does not.** The rules are learned from the *labeled* training templates; LogBERT is trained on normal data only. B4 also relies on the alert vocabulary staying similar over time, which held here (80 rules give the best test F1 of 0.988; the sensitivity curve is figure 9) but is not guaranteed.
* **The window label comes from single messages.** A window is anomalous if it holds one alert-tagged message, and the tag is a property of that message. A method that classifies messages has an advantage over one that must judge a whole sequence.
* **One dataset, one chronological split, one run (seed 42).** There are no confidence intervals; differences of a few points between methods should not be over-read.
* **The language model is a 4-bit quantised 7B model** running with temperature 0. Another model or a larger one could change B1 to B3.

## 9. How to reproduce

From the `NLP/` folder, with the data downloaded (`make data`) and preprocessed (`make preprocess`):

| What | Command |
|---|---|
| train LogBERT | `python scripts/train_logbert.py --config configs/bgl_cpu.yaml` |
| score A and A+ (tunes g and r) | `python scripts/evaluate_logbert.py --config configs/bgl_cpu.yaml` |
| write the rules | `python scripts/induce_rules.py --config configs/bgl_cpu.yaml` |
| run B1 to B4 | `python scripts/run_rule_guided.py --config configs/bgl_cpu.yaml` |
| tables and analysis | `python scripts/compare.py --config configs/bgl_cpu.yaml` and `python scripts/analyze.py --config configs/bgl_cpu.yaml` |
| the figures | `python scripts/make_figures.py --config configs/bgl_cpu.yaml` |

Every language-model request of the run is in `results/llm_cache/`, so approach B can be replayed without a model (see `results/README.md`).
`results/logbert/model.pt` is the trained LogBERT checkpoint, and `make demo` opens a small web page where you paste log lines and see how each method judges them.